# QED-SF-TDA — spin-flip Pauli–Fierz response (ethylene & CH₂)

Interactive checks for the **collinear QED-SF-TDA** path in CasidaPy:

1. **Electronic SF-TDA** on a high-spin UKS reference (Route A, hybrid XC)
2. **Δd dipole-difference** coupling (not the spin-forbidden ⟨α|r|β⟩)
3. **QED-SF-TDA** matrix: SF singles ⊗ {0,1} photons
4. **λ → 0** limit (eigenvalues = {ω_SF} ∪ {ω_SF + ω_c})
5. **λ-scan** and a short **torsion scan** on twisted ethylene

Molecules:

- **CH₂ triplet** — small / fast sanity check
- **Ethylene at 90° torsion** — diradicaloid SF showcase

Level A: ordinary UKS ground state; cavity terms enter the response only (no QED-SCF).

## Prerequisites

```bash
pip install pyscf matplotlib
# from the casidapy repo root:
pip install -e .
```

Defaults use `sto-3g` + `bhandhlyp` so cells finish in minutes on a login node.

In [ ]:
%matplotlib inline

import time
import warnings

import numpy as np
import matplotlib.pyplot as plt

from pyscf import gto, dft

from casidapy import (
    extract_sf_gto_kernel,
    run_casida,
    QEDOptions,
    solve_qed_sf_tda,
    scan_qed_sf_lambda,
    build_qed_sf_tda_matrix,
)
from casidapy.utils.qed import sf_dipole_difference_matrix
from casidapy.kernels.gto import GTOKernel

HA_TO_EV = 27.211386245988
XC = "bhandhlyp"
BASIS = "sto-3g"
print(f"defaults: XC={XC}, basis={BASIS}")

## Helpers

Geometries and a small UKS + SF-kernel builder (reuse `mf` to avoid repeating SCF).

In [ ]:
CH2_ATOM = """
C  0.000000  0.000000  0.000000
H  0.000000  0.000000  1.080000
H  1.000000  0.000000 -0.400000
"""

def ethylene_atom(twist_deg: float = 90.0) -> str:
    """Ethylene with one CH₂ twisted by ``twist_deg`` about the C=C (x) axis."""
    # Planar reference: C=C along x; H's in xy. Twist the +x methylene about x.
    cc = 1.339
    ch = 1.086
    ang = np.deg2rad(121.3)
    # Local methylene: H offsets from C in the molecular frame before twist
    hx = -ch * np.cos(np.pi - ang)
    hy = ch * np.sin(np.pi - ang)
    # Left CH₂ (untwisted, in xy)
    c0 = np.array([0.0, 0.0, 0.0])
    c1 = np.array([cc, 0.0, 0.0])
    h00 = c0 + np.array([hx, hy, 0.0])
    h01 = c0 + np.array([hx, -hy, 0.0])
    # Right CH₂: start in xy, then rotate about x by twist
    t = np.deg2rad(twist_deg)
    R = np.array([
        [1.0, 0.0, 0.0],
        [0.0, np.cos(t), -np.sin(t)],
        [0.0, np.sin(t), np.cos(t)],
    ])
    h10 = c1 + R @ np.array([-hx, hy, 0.0])
    h11 = c1 + R @ np.array([-hx, -hy, 0.0])
    atoms = [("C", c0), ("C", c1), ("H", h00), ("H", h01), ("H", h10), ("H", h11)]
    return "\n".join(f"{s}  {r[0]:.6f}  {r[1]:.6f}  {r[2]:.6f}" for s, r in atoms)


def build_sf_kernel(atom: str, spin: int = 2, basis: str = BASIS, xc: str = XC,
                    verbose: bool = False):
    """Converge UKS and return a ready SF GTOKernel (+ mf)."""
    mol = gto.M(atom=atom, basis=basis, spin=spin, charge=0,
                verbose=4 if verbose else 0)
    mf = dft.UKS(mol)
    mf.xc = xc
    mf.grids.level = 1
    t0 = time.time()
    e = mf.kernel()
    print(f"UKS ({xc}/{basis}, spin={spin}) E = {e:.8f} Ha  ({time.time()-t0:.1f}s)")
    kernel, opts = extract_sf_gto_kernel(
        mol, xc=xc, use_df=False, mf=mf, n_states=8, verbose=verbose,
    )
    kernel.setup(tda=True)
    print(
        f"SF manifold: {kernel.n_occ} α-occ × {kernel.n_unocc} β-virt "
        f"= {kernel.n_trans} transitions"
    )
    return mol, mf, kernel, opts


def print_qed_sf(res, nstates=6):
    n = min(nstates, len(res.omega))
    print(f"{'state':>6}  {'ω (eV)':>10}  {'|m|²':>8}")
    print("-" * 28)
    for i in range(n):
        print(f"{i+1:6d}  {res.omega[i]*HA_TO_EV:10.4f}  {res.photon_frac[i]:8.4f}")

## 1. CH₂ triplet — electronic SF-TDA

Bare spin-flip spectrum (no cavity). Oscillator strengths vanish (dipole-forbidden).

In [ ]:
mol_ch2, mf_ch2, kern_ch2, opts_ch2 = build_sf_kernel(CH2_ATOM)
opts_ch2.solver_method = "davidson"
opts_ch2.n_states = 6
res_sf = run_casida(kern_ch2, opts_ch2)
print("\nBare SF-TDA (CH₂):")
for i, w in enumerate(res_sf.omega[:6]):
    print(f"  P{i}: {w*HA_TO_EV:8.3f} eV   f={res_sf.f[i]:.2e}")
assert np.allclose(res_sf.f[:6], 0.0, atol=1e-10)

## 2. λ → 0 limit (QED-SF recovers bare SF ⊕ shifted copy)

With `λ = 0`, the QED-SF matrix is block-diagonal:

$$\mathrm{spec}(M)=\{\omega_{\mathrm{SF}}\}\cup\{\omega_{\mathrm{SF}}+\omega_c\}$$

In [ ]:
omega_c = 0.15  # Ha ≈ 4.08 eV
A = kern_ch2._K + np.diag(kern_ch2.diagonal_dE())
w_sf = np.sort(np.linalg.eigvalsh(A))

res0 = solve_qed_sf_tda(
    kern_ch2,
    lam_vec=(0.0, 0.0, 0.0),
    omega_c=omega_c,
    nstates=2 * kern_ch2.n_trans,
)
expected = np.sort(np.concatenate([w_sf, w_sf + omega_c]))
err = np.max(np.abs(res0.omega - expected))
print(f"λ=0 max |ω_QED − ω_ref| = {err:.3e} Ha")
assert err < 1e-8

# Each root should be purely electronic or purely photonic
pure = (res0.photon_frac < 1e-8) | (res0.photon_frac > 1.0 - 1e-8)
print(f"pure 0/1 photon character: {np.sum(pure)}/{len(pure)} roots")
assert np.all(pure)

## 3. Δd matrix structure

Slater–Condon one-body elements on the SF manifold:

$$\langle\Phi_{i\alpha}^{a\beta}|\mathbf{d}|\Phi_{j\alpha}^{b\beta}\rangle
= \delta_{ij}\,d_{a\beta,b\beta}-\delta_{ab}\,d_{i\alpha,j\alpha}$$

Contracted with `λ` → `Δ = I ⊗ Q_vv − Q_oo ⊗ I`.

In [ ]:
lam = np.array([0.0, 0.0, 0.05])
delta = sf_dipole_difference_matrix(kern_ch2, lam)
print(f"Δ shape = {delta.shape}, symmetric = {np.allclose(delta, delta.T)}")
print(f"||Δ||_F = {np.linalg.norm(delta):.4f}")
assert np.allclose(delta, delta.T, atol=1e-10)

M = build_qed_sf_tda_matrix(kern_ch2, lam, omega_c=0.1)
print(f"QED-SF matrix shape = {M.shape}  (expect 2 n_trans = {2*kern_ch2.n_trans})")
assert M.shape == (2 * kern_ch2.n_trans, 2 * kern_ch2.n_trans)
assert np.allclose(M, M.T, atol=1e-10)

## 4. Twisted ethylene (90°) — SF + QED-SF spectrum

At 90° torsion the π system is diradicaloid; SF from the triplet reference is the natural route.

In [ ]:
mol_et, mf_et, kern_et, opts_et = build_sf_kernel(ethylene_atom(90.0))
opts_et.solver_method = "davidson"
opts_et.n_states = 8
bare = run_casida(kern_et, opts_et)

print("\nBare SF-TDA (ethylene 90°):")
for i, w in enumerate(bare.omega[:8]):
    print(f"  P{i}: {w*HA_TO_EV:8.3f} eV")

# Tune cavity near the lowest SF root
omega_c_et = float(bare.omega[0])
print(f"\nCavity tuned to lowest SF root: ω_c = {omega_c_et*HA_TO_EV:.3f} eV")

qed_opts = QEDOptions(
    lam_scalar=0.05,
    polarization=(0.0, 0.0, 1.0),
    omega_c=omega_c_et,
    nstates=8,
)
res_qed = solve_qed_sf_tda(kern_et, options=qed_opts)
print("\nQED-SF-TDA (λ=0.05, z-pol):")
print_qed_sf(res_qed, nstates=8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)

axes[0].vlines(bare.omega[:8] * HA_TO_EV, 0, 1, colors="C0", lw=1.6)
axes[0].set_title("Bare SF-TDA")
axes[0].set_xlabel("ω (eV)")
axes[0].set_ylabel("arb.")
axes[0].set_ylim(0, 1.15)

sc = axes[1].scatter(
    res_qed.omega * HA_TO_EV,
    np.ones_like(res_qed.omega),
    c=res_qed.photon_frac,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    s=60,
    zorder=3,
)
axes[1].vlines(res_qed.omega * HA_TO_EV, 0, 1, colors="0.7", lw=1.0, zorder=1)
axes[1].axvline(omega_c_et * HA_TO_EV, color="k", ls="--", lw=1, label="ω_c")
axes[1].set_title("QED-SF-TDA (color = photon frac.)")
axes[1].set_xlabel("ω (eV)")
axes[1].legend(loc="upper right", fontsize=8)
fig.colorbar(sc, ax=axes[1], label="|1-ph|²")
fig.suptitle("Ethylene 90° torsion — bhandhlyp/sto-3g", y=1.02)
fig.tight_layout()
plt.show()

## 5. λ-scan at fixed 90° geometry

Electronic SF `A'` is built once; only `Δ ∝ λ` is rebuilt. Branches tracked by Hungarian overlap on the full `2 n_trans` eigenvectors.

In [ ]:
lams = np.linspace(0.0, 0.08, 9)
scan = scan_qed_sf_lambda(
    kern_et,
    lam_scalars=lams,
    polarization=(0.0, 0.0, 1.0),
    omega_c=omega_c_et,
    nstates=6,
    track=True,
)
omega_t = scan["omega_tracked"] * HA_TO_EV
phot_t = scan["photon_frac_tracked"]

fig, ax = plt.subplots(figsize=(6.5, 4.0))
for k in range(omega_t.shape[1]):
    ax.plot(lams, omega_t[:, k], "-o", ms=4, label=f"P{k}")
ax.axhline(omega_c_et * HA_TO_EV, color="k", ls="--", lw=1, label="ω_c")
ax.set_xlabel("λ (a.u.)")
ax.set_ylabel("ω (eV)")
ax.set_title("QED-SF λ-scan — ethylene 90° (tracked)")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.2))
for k in range(phot_t.shape[1]):
    ax.plot(lams, phot_t[:, k], "-o", ms=3, label=f"P{k}")
ax.set_xlabel("λ (a.u.)")
ax.set_ylabel("photon fraction")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Photonic character along λ-scan")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

## 6. Torsion PES (few angles) with fixed ω_c and λ

Short scan around the diradical region. `ω_c` is held fixed (Ha) across the scan.

**Tracking:** use `track_states(..., method="energy")` for geometry / torsion scans. Eigenvector-overlap tracking is only valid for λ-scans at **fixed** geometry — across torsion the SF config basis rides on changing MOs, so raw `X` overlaps scramble branches.


In [ ]:
angles = np.array([60.0, 75.0, 90.0, 105.0, 120.0])
lam_fixed = 0.05
pol = (0.0, 0.0, 1.0)
nstates_pes = 5

results_pes = []
bare_pes = []
for ang in angles:
    print(f"\n=== twist = {ang:.0f}° ===")
    _, _, kern, _ = build_sf_kernel(ethylene_atom(ang))
    A = kern._K + np.diag(kern.diagonal_dE())
    w_bare = np.sort(np.linalg.eigvalsh(A))[:nstates_pes]
    bare_pes.append(w_bare)
    r = solve_qed_sf_tda(
        kern,
        lam_vec=np.asarray(pol, float) * lam_fixed,
        omega_c=omega_c_et,
        nstates=nstates_pes,
    )
    results_pes.append(r)
    print_qed_sf(r, nstates=nstates_pes)

from casidapy.utils.qed import track_states

# Geometry scan: energy (+ photon-frac) matching — NOT overlap.
# Overlap tracking is for λ-scans at fixed geometry only (shared MO basis).
omega_tr, phot_tr = track_states(
    results_pes, method="energy", photon_weight=0.05,
)
bare_pes = np.asarray(bare_pes)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
for k in range(min(3, bare_pes.shape[1])):
    axes[0].plot(angles, bare_pes[:, k] * HA_TO_EV, "--o", ms=4, label=f"SF P{k}")
for k in range(omega_tr.shape[1]):
    axes[1].plot(angles, omega_tr[:, k] * HA_TO_EV, "-o", ms=4, label=f"QED P{k}")
for k in range(omega_tr.shape[1]):
    axes[1].scatter(
        angles, omega_tr[:, k] * HA_TO_EV, c=phot_tr[:, k],
        cmap="coolwarm", vmin=0, vmax=1, s=28, zorder=3,
    )
axes[1].axhline(omega_c_et * HA_TO_EV, color="k", ls=":", lw=1, label="ω_c")
axes[0].set_title("Bare SF-TDA")
axes[1].set_title(f"QED-SF-TDA (λ={lam_fixed}, energy-tracked)")
for ax in axes:
    ax.set_xlabel("torsion (deg)")
    ax.set_ylabel("ω (eV)")
    ax.legend(fontsize=8)
fig.suptitle("Ethylene torsion — adiabatic branches (energy match)", y=1.02)
fig.tight_layout()
plt.show()


## Notes / limits

- **Route A SF**: exchange-only; needs a hybrid (`bhandhlyp`, `pbe0`, …).
- **TDA only** for SF and QED-SF.
- **Δd coupling** lives in same-spin MO dipoles; absorption oscillator strengths from the triplet reference remain zero.
- **DSE / coherent-state** are off in QED-SF (bilinear / JC-like form). Closed-shell `solve_qed_tda` still has DSE+CS.
- For production: larger basis, denser torsion grid, and compare to literature QED-SF-CIS where available.

CLI twin: `python scripts/run_sf_tda.py --qed --molecule twisted-ethylene ...`